In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Carregando os dados
df_curto = pd.read_excel("curto_ia_regex.xlsx")
df_curto.head()

# Análise Exploratória de Dados

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configurações de estilo
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


## Mapas Coroplético — Processos por Comarca (Estado de SP)

Visualização geográfica dos processos sobre o mapa do estado de São Paulo.
Cada município é colorido de acordo com a métrica selecionada.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Carregar GeoJSON dos municípios de SP
with open(Path('geodata') / 'SP.json', 'r', encoding='utf-8') as f:
    sp_geojson = json.load(f)

# Normalizar nomes para matching
def _normalizar_nome_geo(nome):
    if pd.isna(nome): return ''
    t = unicodedata.normalize('NFKD', str(nome)).encode('ascii', 'ignore').decode('ascii')
    return t.strip().lower()

# Mapear nome do município -> GEOCODIGO
comarca_map = {}
for feat in sp_geojson['features']:
    comarca_map[_normalizar_nome_geo(feat['properties']['NOME'])] = feat['properties']['GEOCODIGO']

# Lista completa de todos os municípios de SP (base sempre desenhada)
sp_all_geocodigos = [feat['properties']['GEOCODIGO'] for feat in sp_geojson['features']]
sp_all_nomes = [feat['properties']['NOME'] for feat in sp_geojson['features']]

# Agregar por comarca
df_mapa = df_curto.groupby('comarca').agg(
    total_processos=('id_processo', 'count'),
    valor_medio_morais=('valor_danos_morais_ia', 'mean'),
).reset_index()

# Taxa de procedência
def _calc_taxa(g):
    total = len(g)
    if total == 0: return 0.0
    favoraveis = g['resultado_julgamento_ia'].isin(['procedente', 'parcialmente procedente']).sum()
    return (favoraveis / total) * 100

taxa = df_curto.groupby('comarca').apply(_calc_taxa, include_groups=False).reset_index()
taxa.columns = ['comarca', 'taxa_procedencia']
df_mapa = df_mapa.merge(taxa, on='comarca', how='left')

# Match comarca -> geocodigo
df_mapa['comarca_norm'] = df_mapa['comarca'].apply(_normalizar_nome_geo)
df_mapa['geocodigo'] = df_mapa['comarca_norm'].map(comarca_map)
df_mapa['valor_medio_morais'] = df_mapa['valor_medio_morais'].round(2)
df_mapa['taxa_procedencia'] = df_mapa['taxa_procedencia'].round(1)

# Estatísticas de match
matched = df_mapa['geocodigo'].notna().sum()
total_comarcas = len(df_mapa)
pct = matched / total_comarcas * 100
print(f'Comarcas mapeadas: {matched}/{total_comarcas} ({pct:.1f}%)')
if total_comarcas - matched > 0:
    sem_match = df_mapa[df_mapa['geocodigo'].isna()]['comarca'].tolist()
    print(f'Sem match ({total_comarcas - matched}):', sem_match)

df_plot = df_mapa[df_mapa['geocodigo'].notna()].copy()


def make_sp_choropleth(df, color_col, color_scale, title, value_label):
    # Camada 1 — base: todos os municípios de SP em cinza escuro neutro
    base = go.Choropleth(
        geojson=sp_geojson,
        locations=sp_all_geocodigos,
        featureidkey='properties.GEOCODIGO',
        z=[0] * len(sp_all_geocodigos),
        colorscale=[[0, '#1f2937'], [1, '#1f2937']],
        showscale=False,
        marker_line_color='rgba(255,255,255,0.35)',
        marker_line_width=0.4,
        hovertext=sp_all_nomes,
        hovertemplate='<b>%{hovertext}</b><br><i>sem processos</i><extra></extra>',
    )

    # Camada 2 — dados: municípios com valores na escala selecionada
    data_layer = go.Choropleth(
        geojson=sp_geojson,
        locations=df['geocodigo'],
        featureidkey='properties.GEOCODIGO',
        z=df[color_col],
        colorscale=color_scale,
        marker_line_color='white',
        marker_line_width=0.9,
        customdata=df[['comarca', 'total_processos', 'valor_medio_morais', 'taxa_procedencia']].values,
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'Processos: %{customdata[1]:,}<br>'
            'Danos morais médios: R$ %{customdata[2]:,.2f}<br>'
            'Procedência: %{customdata[3]:.1f}%'
            '<extra></extra>'
        ),
        colorbar=dict(title=value_label, thickness=14, len=0.75, outlinewidth=0),
    )

    fig = go.Figure(data=[base, data_layer])
    fig.update_geos(
        fitbounds='geojson',
        visible=False,
        projection_type='mercator',
        bgcolor='rgba(0,0,0,0)',
    )
    fig.update_layout(
        title=dict(text=title, x=0.02, xanchor='left'),
        margin={'r': 0, 't': 50, 'l': 0, 'b': 0},
        height=620,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
    )
    return fig


In [ ]:
# Mapa 1: Volume de processos por comarca
fig1 = make_sp_choropleth(
    df_plot,
    color_col='total_processos',
    color_scale='YlOrRd',
    title='Volume de Processos por Comarca — Estado de São Paulo',
    value_label='Nº Processos',
)
fig1.show()


In [ ]:
# Mapa 2: Valor médio de danos morais por comarca
fig2 = make_sp_choropleth(
    df_plot,
    color_col='valor_medio_morais',
    color_scale='Purples',
    title='Valor Médio de Danos Morais por Comarca — Estado de São Paulo',
    value_label='Valor Médio (R$)',
)
fig2.show()


In [ ]:
# Mapa 3: Taxa de procedência por comarca
fig3 = make_sp_choropleth(
    df_plot,
    color_col='taxa_procedencia',
    color_scale='Greens',
    title='Taxa de Procedência por Comarca — Estado de São Paulo',
    value_label='Procedência (%)',
)
fig3.show()


### 1. Proporção de Contato Prévio

## Qual a porcentagem geral de processos em que o autor tentou resolver o problema antes de judicializar?

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df_curto, x='contato_previo_banco_ia', palette='viridis')
plt.title('Proporção de Casos com Contato Prévio ao Banco', fontsize=14)
plt.xlabel('Contato Prévio?', fontsize=12)
plt.ylabel('Quantidade de Processos', fontsize=12)

# Adicionar porcentagens acima das barras
total = len(df_curto['contato_previo_banco_ia'].dropna())
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height/total:.1%}', (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=11)
plt.show()


### 2. Canais de Contato Mais Utilizados

## Quando o cliente procura o banco, por onde ele vai?

In [ ]:
canais = df_curto[df_curto['canal_contato_ia'] != 'não identificado']

plt.figure(figsize=(10, 5))
ax = sns.countplot(data=canais, y='canal_contato_ia', order=canais['canal_contato_ia'].value_counts().index, palette='magma')
plt.title('Canais de Contato Prévio Mais Utilizados', fontsize=14)
plt.xlabel('Quantidade de Processos', fontsize=12)
plt.ylabel('Canal de Contato', fontsize=12)
plt.show()


### 3. Visão Geral dos Resultados e Valores

## Como se distribuem os resultados das sentenças e as indenizações?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Resultados
sns.countplot(data=df_curto, y='resultado_julgamento_ia', 
              order=df_curto['resultado_julgamento_ia'].value_counts().index, 
              palette='coolwarm', ax=axes[0])
axes[0].set_title('Distribuição dos Resultados dos Julgamentos', fontsize=14)
axes[0].set_xlabel('Quantidade', fontsize=12)
axes[0].set_ylabel('Resultado', fontsize=12)

# Valores de Danos Morais (apenas > 0)
danos_morais_positivos = df_curto[df_curto['valor_danos_morais_ia'] > 0]
sns.histplot(danos_morais_positivos['valor_danos_morais_ia'], bins=15, kde=True, color='purple', ax=axes[1])
axes[1].set_title('Distribuição dos Valores de Danos Morais (Condenações > R$ 0)', fontsize=14)
axes[1].set_xlabel('Valor (R$)', fontsize=12)
axes[1].set_ylabel('Frequência', fontsize=12)

plt.tight_layout()
plt.show()


### 4. Contato Prévio vs. Probabilidade de Vitória

## Clientes que tentaram resolver administrativamente têm taxa de procedência maior?

In [ ]:
contato_resultado = pd.crosstab(df_curto['contato_previo_banco_ia'], df_curto['resultado_julgamento_ia'], normalize='index') * 100

ax = contato_resultado.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Set2')
plt.title('Resultado do Julgamento por Contato Prévio (%)', fontsize=14)
plt.xlabel('Houve Contato Prévio?', fontsize=12)
plt.ylabel('Proporção (%)', fontsize=12)
plt.legend(title='Resultado', bbox_to_anchor=(1.05, 1), loc='upper left')

# Adicionar os valores nas barras
for c in ax.containers:
    labels = [f'{v.get_height():.1f}%' if v.get_height() > 0 else '' for v in c]
    ax.bar_label(c, labels=labels, label_type='center', fontsize=10)
    
plt.tight_layout()
plt.show()


### 5. Contato Prévio vs. Valores de Indenização (Danos Morais)

## Juízes aplicam indenizações maiores quando o cliente comprova que tentou contato?

In [ ]:
plt.figure(figsize=(10, 6))
# Filtrar apenas quem ganhou danos morais
condenados_morais = df_curto[df_curto['valor_danos_morais_ia'] > 0]

sns.boxplot(data=condenados_morais, x='contato_previo_banco_ia', y='valor_danos_morais_ia', palette='Set3')
plt.title('Distribuição do Valor de Danos Morais por Contato Prévio (Apenas Condenações > 0)', fontsize=14)
plt.xlabel('Houve Contato Prévio?', fontsize=12)
plt.ylabel('Valor de Danos Morais (R$)', fontsize=12)
plt.show()

# Imprimir as médias
medias = condenados_morais.groupby('contato_previo_banco_ia')['valor_danos_morais_ia'].mean()
print("Média de Danos Morais (quando há condenação):")
print(medias)


### 6. Impacto do Canal de Contato no Resultado

## Acionar o Procon/Reclame Aqui gera maior probabilidade de vitória do que o SAC?

In [ ]:
plt.figure(figsize=(10, 6))
canal_resultado = pd.crosstab(canais['canal_contato_ia'], canais['resultado_julgamento_ia'], normalize='index') * 100

sns.heatmap(canal_resultado, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={'label': 'Proporção (%)'})
plt.title('Probabilidade de Resultado por Canal de Contato Prévio', fontsize=14)
plt.ylabel('Canal de Contato', fontsize=12)
plt.xlabel('Resultado do Julgamento', fontsize=12)
plt.show()


### 7. Tipo de Ação vs. Contato Prévio

## Quais os tipos de ação com maior volume de tentativa prévia de acordo?

In [ ]:
plt.figure(figsize=(10, 6))
tipo_contato = pd.crosstab(df_curto['tipo_acao_ia'], df_curto['contato_previo_banco_ia'], normalize='index') * 100

ax = tipo_contato.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Pastel1')
plt.title('Proporção de Contato Prévio por Tipo de Ação (%)', fontsize=14)
plt.xlabel('Tipo de Ação', fontsize=12)
plt.ylabel('Proporção (%)', fontsize=12)
plt.legend(title='Houve Contato?', bbox_to_anchor=(1.05, 1), loc='upper left')

for c in ax.containers:
    labels = [f'{v.get_height():.1f}%' if v.get_height() > 0 else '' for v in c]
    ax.bar_label(c, labels=labels, label_type='center', fontsize=10)
    
plt.tight_layout()
plt.show()


### 8. Boletim de Ocorrência vs Probabilidade de Vitória

## Fazer B.O. ajuda a ganhar a causa?

In [ ]:
plt.figure(figsize=(10, 6))
bo_resultado = pd.crosstab(df_curto['boletim_de_ocorrencia_ia'], df_curto['resultado_julgamento_ia'], normalize='index') * 100

ax = bo_resultado.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Spectral')
plt.title('Resultado do Julgamento por Existência de B.O. (%)', fontsize=14)
plt.xlabel('Fez Boletim de Ocorrência?', fontsize=12)
plt.ylabel('Proporção (%)', fontsize=12)
plt.legend(title='Resultado', bbox_to_anchor=(1.05, 1), loc='upper left')

for c in ax.containers:
    labels = [f'{v.get_height():.1f}%' if v.get_height() > 0 else '' for v in c]
    ax.bar_label(c, labels=labels, label_type='center', fontsize=10)
    
plt.tight_layout()
plt.show()


### 9. Atribuição de Culpa vs Contato Prévio

## Se houve contato, a culpa cai mais em cima do banco?

In [ ]:
plt.figure(figsize=(10, 6))
culpa_contato = pd.crosstab(df_curto['culpa_atribuida_ia'], df_curto['contato_previo_banco_ia'], normalize='columns') * 100

ax = culpa_contato.T.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='tab20')
plt.title('Distribuição de Culpa Atribuída de Acordo com Contato Prévio (%)', fontsize=14)
plt.xlabel('Houve Contato Prévio?', fontsize=12)
plt.ylabel('Proporção (%)', fontsize=12)
plt.legend(title='Culpa Atribuída', bbox_to_anchor=(1.05, 1), loc='upper left')

for c in ax.containers:
    labels = [f'{v.get_height():.1f}%' if v.get_height() > 0 else '' for v in c]
    ax.bar_label(c, labels=labels, label_type='center', fontsize=10)
    
plt.tight_layout()
plt.show()


### 10. Tabela Resumo (Relatório 2)

In [ ]:
# a) Porcentagem de processos com contato prévio
pct_contato = (df_curto['contato_previo_banco_ia'] == 'sim').mean() * 100

# b) Médias no subgrupo que fez contato prévio
subgrupo_contato = df_curto[df_curto['contato_previo_banco_ia'] == 'sim']

media_materiais_com_zeros = subgrupo_contato['valor_danos_materiais_ia'].mean()
media_morais_com_zeros = subgrupo_contato['valor_danos_morais_ia'].mean()

subgrupo_materiais_condenados = subgrupo_contato[subgrupo_contato['valor_danos_materiais_ia'] > 0]
subgrupo_morais_condenados = subgrupo_contato[subgrupo_contato['valor_danos_morais_ia'] > 0]

media_materiais_somente_condenados = subgrupo_materiais_condenados['valor_danos_materiais_ia'].mean()
media_morais_somente_condenados = subgrupo_morais_condenados['valor_danos_morais_ia'].mean()

print(f"a) Porcentagem com contato prévio: {pct_contato:.2f}%")
print(f"b) Média de danos materiais (incluindo zeros): R$ {media_materiais_com_zeros:.2f}")
print(f"c) Média de danos morais (incluindo zeros): R$ {media_morais_com_zeros:.2f}")
print(f"d) Média de danos materiais (somente condenados): R$ {media_materiais_somente_condenados:.2f}")
print(f"e) Média de danos morais (somente condenados): R$ {media_morais_somente_condenados:.2f}")

# Montar o dataframe e salvar no relatorio.xlsx
df_relatorio = pd.DataFrame([{
    "pct_contato_previo": pct_contato,
    "media_danos_materiais": media_materiais_com_zeros,
    "media_danos_morais": media_morais_com_zeros,
    "media_danos_materiais_somente_condenados": media_materiais_somente_condenados,
    "media_danos_morais_somente_condenados": media_morais_somente_condenados,
}])

df_relatorio.to_excel("relatorio.xlsx", index=False)
print("\nArquivo relatorio.xlsx gerado com sucesso!")

# Modelo Inferencial — Efeito do Contato Prévio com o Banco

**Três modelos inferenciais:**

| # | Modelo | Pergunta |
|---|--------|----------|
| A | Logit em **procedência** | O contato prévio aumenta a chance de o autor vencer? |
| B | OLS em **log(valor de danos morais)** | Em quem é condenado, o contato prévio muda o valor da condenação? |
| C | Logit em **contato prévio** | Qual o *perfil* de caso associado a ter havido contato prévio? |

Cada modelo é estimado em **3 especificações** (bivariada → com controles → com comarca) para isolar o efeito do contato dos confundidores (tipo de ação, rito, comarca, etc.). A estabilidade do coeficiente do contato entre as specs é o teste real de robustez da associação.


In [9]:
# === Configuração da análise inferencial ===
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

RANDOM_STATE = 42

# Variáveis de interesse
VAR_TRATAMENTO = 'contato_previo_banco_ia'      # 'sim' / 'não'
NIVEL_BASE     = 'não'                           # referência para o coeficiente

# Controles do caso (não derivados da sentença)
CONTROLES_CASO = [
    'tipo_acao_ia',
    'rito_processual_ia',
    'boletim_de_ocorrencia_ia',
    'justica_gratuita_ia',
    'mencao_reclame_aqui_ia',
]

# Variáveis de alta cardinalidade — agrupadas (raras viram "outras")
ALTA_CARDIN = ['comarca', 'assunto']
MIN_FREQ_GRUPO = 20  # categorias com < N casos viram 'outras'

# Outcomes
OUTCOME_PROCED   = 'procedente'                 # binária derivada de resultado_julgamento_ia
OUTCOME_VL_MORAL = 'valor_danos_morais_ia'      # numérico (R$)


In [10]:
# === Preparação do dataset inferencial ===
df_inf = df_curto.copy()

# 1) Tratamento: contato prévio em 'sim'/'não'
df_inf = df_inf[df_inf[VAR_TRATAMENTO].isin(['sim', 'não'])].copy()

# 2) Outcome A — procedência (vitória do autor): procedente ou parcialmente procedente = 1
def _proced(r):
    if r in ['procedente', 'parcialmente procedente']: return 1
    if r in ['improcedente']:                          return 0
    return np.nan  # extinto / não identificado: fora desse modelo

df_inf[OUTCOME_PROCED] = df_inf['resultado_julgamento_ia'].apply(_proced)

# 3) Outcome B — log do valor de danos morais (somente em casos com condenação > 0)
df_inf['ln_morais']    = np.where(
    df_inf[OUTCOME_VL_MORAL] > 0,
    np.log(df_inf[OUTCOME_VL_MORAL].fillna(0).astype(float)),
    np.nan,
)
df_inf['condenado_moral'] = (df_inf[OUTCOME_VL_MORAL] > 0).astype(int)

# 4) Categóricas — NaN explícito + agrupar raras
for col in CONTROLES_CASO + [VAR_TRATAMENTO]:
    df_inf[col] = df_inf[col].fillna('desconhecido').astype(str)

for col in ALTA_CARDIN:
    df_inf[col] = df_inf[col].fillna('desconhecido').astype(str)
    counts = df_inf[col].value_counts()
    freq   = counts[counts >= MIN_FREQ_GRUPO].index
    df_inf[f'{col}_grp'] = df_inf[col].where(df_inf[col].isin(freq), 'outras')

print(f'Linhas totais: {len(df_curto):,}')
print(f'Linhas com tratamento válido (sim/não): {len(df_inf):,}\n')

print('Outcome A — procedência (descartado extinto/não identificado):')
print(df_inf[OUTCOME_PROCED].value_counts(dropna=False).rename('n'))
print()
print('Outcome B — condenado em danos morais > 0:')
print(df_inf['condenado_moral'].value_counts().rename('n'))
print()
print(f'Comarcas distintas após agrupamento: {df_inf["comarca_grp"].nunique()}')
print(f'Assuntos distintos após agrupamento: {df_inf["assunto_grp"].nunique()}')


NameError: name 'df_curto' is not defined

In [ ]:
# === Helpers para comparar especificações ===
def _termo_tratamento():
    """Nome do termo do tratamento dentro do output do statsmodels."""
    return f'C({VAR_TRATAMENTO}, Treatment("{NIVEL_BASE}"))[T.sim]'


def resumo_termo(modelo, termo, exponentiate=False):
    """Coef, std err, p-value, IC 95% para um termo específico.

    Se exponentiate=True (Logit), retorna OR e IC do OR.
    """
    if termo not in modelo.params.index:
        return None
    coef    = modelo.params[termo]
    se      = modelo.bse[termo]
    p       = modelo.pvalues[termo]
    ci_low, ci_high = modelo.conf_int().loc[termo]
    out = {
        'coef':       coef,
        'std_err':    se,
        'p_value':    p,
        'CI95_low':   ci_low,
        'CI95_high':  ci_high,
        'n_obs':      int(modelo.nobs),
    }
    if exponentiate:
        out.update({
            'OR':         np.exp(coef),
            'OR_CI95_lo': np.exp(ci_low),
            'OR_CI95_hi': np.exp(ci_high),
        })
    return out


def tabela_specs(modelos_dict, termo, exponentiate=False):
    """Tabela comparando o mesmo termo em várias especificações."""
    rows = []
    for nome, m in modelos_dict.items():
        r = resumo_termo(m, termo, exponentiate=exponentiate)
        if r is None:
            continue
        rows.append({'spec': nome, **r})
    df = pd.DataFrame(rows).set_index('spec')
    return df.round(4)


def estrela(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.10:  return '.'
    return ''


In [ ]:
# === Modelo A — Logit em procedência ===
# H1: o contato prévio com o banco aumenta a probabilidade de o autor vencer?

df_a = df_inf.dropna(subset=[OUTCOME_PROCED]).copy()
df_a[OUTCOME_PROCED] = df_a[OUTCOME_PROCED].astype(int)

f_base    = f'{OUTCOME_PROCED} ~ C({VAR_TRATAMENTO}, Treatment("{NIVEL_BASE}"))'
f_controles = f_base + ' + ' + ' + '.join(f'C({c})' for c in CONTROLES_CASO)
f_completa  = f_controles + ' + C(comarca_grp) + C(assunto_grp)'

modelos_A = {
    '1_bivariado':  smf.logit(f_base,      data=df_a).fit(disp=0),
    '2_controles':  smf.logit(f_controles, data=df_a).fit(disp=0),
    '3_completo':   smf.logit(f_completa,  data=df_a).fit(disp=0),
}

print('Modelo A — Logit em procedência (vitória do autor)')
print(f'n = {len(df_a):,}\n')

tab_A = tabela_specs(modelos_A, _termo_tratamento(), exponentiate=True)
print('Efeito do contato prévio (vs "não") em cada especificação:')
print(tab_A)
print()
print('Leitura: OR > 1 → contato prévio aumenta chance de procedência; OR < 1 → reduz.')
print('Significância: *** p<0.001 | ** p<0.01 | * p<0.05 | . p<0.10')


In [ ]:
# === Modelo B — OLS em log(valor de danos morais), em condenados ===
# H2: entre quem é condenado em danos morais, o contato prévio muda o valor?

df_b = df_inf.dropna(subset=['ln_morais']).copy()

f_base      = f'ln_morais ~ C({VAR_TRATAMENTO}, Treatment("{NIVEL_BASE}"))'
f_controles = f_base + ' + ' + ' + '.join(f'C({c})' for c in CONTROLES_CASO)
f_completa  = f_controles + ' + C(comarca_grp) + C(assunto_grp)'

# HC3 = erros-padrão robustos a heterocedasticidade
modelos_B = {
    '1_bivariado':  smf.ols(f_base,      data=df_b).fit(cov_type='HC3'),
    '2_controles':  smf.ols(f_controles, data=df_b).fit(cov_type='HC3'),
    '3_completo':   smf.ols(f_completa,  data=df_b).fit(cov_type='HC3'),
}

print('Modelo B — OLS em log(valor de danos morais), somente condenações > 0')
print(f'n = {len(df_b):,}\n')

tab_B = tabela_specs(modelos_B, _termo_tratamento(), exponentiate=False)
print('Efeito do contato prévio (vs "não") em log(R$) em cada spec:')
print(tab_B)
print()
print('Leitura log-linear: % de mudança no valor ≈ (exp(coef) - 1) × 100.')


In [ ]:
# === Modelo C — Logit no contato prévio (perfil do caso) ===
# Qual o perfil de caso associado a ter havido contato prévio com o banco?

df_c = df_inf.copy()
df_c['contato_sim'] = (df_c[VAR_TRATAMENTO] == 'sim').astype(int)

# Aqui o contato é o OUTCOME — predito por características do caso
f_perfil = 'contato_sim ~ ' + ' + '.join(f'C({c})' for c in CONTROLES_CASO) + ' + C(comarca_grp) + C(assunto_grp)'
modelo_C = smf.logit(f_perfil, data=df_c).fit(disp=0)

# Top termos por |z-score| (significância da associação)
def top_termos(modelo, k=20, exclui_intercepto=True):
    df = pd.DataFrame({
        'coef':    modelo.params,
        'OR':      np.exp(modelo.params),
        'std_err': modelo.bse,
        'z':       modelo.tvalues,
        'p_value': modelo.pvalues,
    })
    df['sig'] = df['p_value'].apply(estrela)
    df['|z|'] = df['z'].abs()
    if exclui_intercepto and 'Intercept' in df.index:
        df = df.drop(index='Intercept')
    return df.sort_values('|z|', ascending=False).head(k).drop(columns='|z|').round(4)

print(f'Modelo C — Logit em contato_previo (n = {int(modelo_C.nobs):,})')
print(f'Pseudo R² (McFadden) = {modelo_C.prsquared:.4f}\n')
print('Top 20 características associadas a ter havido contato prévio:')
print(top_termos(modelo_C, k=20))


In [ ]:
# === Resumo de hipóteses — leitura inferencial direta ===

def relato_logit(modelos, termo, label_h):
    m = modelos['3_completo']
    r = resumo_termo(m, termo, exponentiate=True)
    if r is None:
        print(f'[{label_h}] termo não encontrado.')
        return
    s = estrela(r['p_value'])
    direc = 'aumenta' if r['OR'] > 1 else 'reduz'
    print(f'[{label_h}]')
    print(f'  OR = {r["OR"]:.2f}  (IC 95%: {r["OR_CI95_lo"]:.2f} – {r["OR_CI95_hi"]:.2f})')
    print(f'  p-value = {r["p_value"]:.4f} {s}')
    print(f'  Leitura: ter feito contato prévio {direc} as chances do desfecho em {abs(r["OR"]-1)*100:.1f}%, mantendo demais variáveis constantes.')
    print(f'  n = {r["n_obs"]:,}')
    print()


def relato_ols(modelos, termo, label_h, unidade='valor'):
    m = modelos['3_completo']
    r = resumo_termo(m, termo, exponentiate=False)
    if r is None:
        print(f'[{label_h}] termo não encontrado.')
        return
    s = estrela(r['p_value'])
    pct = (np.exp(r['coef']) - 1) * 100
    pct_lo = (np.exp(r['CI95_low']) - 1) * 100
    pct_hi = (np.exp(r['CI95_high']) - 1) * 100
    direc = 'maior' if pct > 0 else 'menor'
    print(f'[{label_h}]')
    print(f'  coef (log) = {r["coef"]:.3f}  (IC 95%: {r["CI95_low"]:.3f} – {r["CI95_high"]:.3f})')
    print(f'  p-value = {r["p_value"]:.4f} {s}')
    print(f'  Leitura: ter feito contato prévio está associado a um {unidade} {abs(pct):.1f}% {direc}, mantendo demais variáveis constantes (IC 95%: {pct_lo:+.1f}% a {pct_hi:+.1f}%).')
    print(f'  n = {r["n_obs"]:,}')
    print()


print('=' * 70)
print('CONCLUSÕES DO MODELO INFERENCIAL')
print('=' * 70)
print()

print('Pergunta central do plano: características do caso e da sentença')
print('associadas a processos com contato prévio ao banco?\n')

relato_logit(modelos_A, _termo_tratamento(),
             'H1 — Contato prévio → probabilidade de procedência')
relato_ols(modelos_B, _termo_tratamento(),
           'H2 — Contato prévio → valor da condenação (danos morais)',
           unidade='valor de condenação')

# H3 — perfil: cite os termos mais associados (z-score)
print('[H3 — Perfil dos casos com contato prévio]')
print('  Top 5 características mais associadas (ranking por |z|):')
top5 = top_termos(modelo_C, k=5)
for idx, row in top5.iterrows():
    direc = '↑' if row['coef'] > 0 else '↓'
    print(f'    {direc} {idx:<60} OR={row["OR"]:.2f}  p={row["p_value"]:.4f} {row["sig"]}')


## Como iterar

- **Adicionar controles:** inclua colunas em `CONTROLES_CASO`. Reexecute as células do Modelo A, B e C.
- **Trocar a referência do tratamento:** mude `NIVEL_BASE` (ex.: `'sim'` para inverter sinal).
- **Mudar o ponto de corte do agrupamento de raros:** ajuste `MIN_FREQ_GRUPO`. Quanto maior, menos categorias entram no modelo.
- **Outras hipóteses:**
  - **Interação `contato_previo × tipo_acao`:** acrescente `+ C(contato_previo_banco_ia):C(tipo_acao_ia)` na fórmula completa e teste se o efeito do contato depende do tipo de ação (efeito heterogêneo).
  - **Modelo de dois estágios para o valor:** primeiro Logit em `condenado_moral` (probabilidade de ser condenado), depois OLS em quem foi condenado (este já está implementado). A combinação dá o efeito médio total.
- **Robustez:**
  - Trocar `cov_type='HC3'` por `cov_type='cluster', cov_kwds={'groups': df_b['comarca_grp']}` para erros-padrão clusterizados por comarca.
  - Estimar com `sm.GLM(..., family=sm.families.Binomial())` para comparar com a Logit.
- **Comparação formal de specs:** `m.compare_lr_test(m_menor)` para teste de razão de verossimilhança entre modelos aninhados.
